# MMF Hierarchical Reconciliation
Auto-generated by mmf-agent.

In [ ]:
%pip install /Workspace/Repos/{full_email}/many-model-forecasting[hierarchical] --quiet

In [ ]:
%restart_python

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"

# One entry per hierarchy level, ordered leaf → root
levels = {levels}

hierarchy_table = "{hierarchy_table}"  # adjacency list: unique_id, level_name, parent_unique_id

freq = "{freq}"  # H | D | W | M
date_col = "{date_col}"
target = "{target}"
reconciliation_method = "{reconciliation_method}"  # BottomUp | TopDown | MiddleOut | MinTrace | ERM
mintrace_method = "{mintrace_method}"  # mint_shrink | wls_struct | wls_var | mint_cov (only used when reconciliation_method=MinTrace)
middle_level = "{middle_level}"  # required only when reconciliation_method=MiddleOut


In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_reconciliation_multilevel

run_reconciliation_multilevel(
    spark=spark,
    levels=levels,
    hierarchy_table=hierarchy_table,
    reconciliation_output=catalog + "." + schema + "." + use_case + "_reconciliation_output",
    freq=freq,
    date_col=date_col,
    target=target,
    method=reconciliation_method,
    mintrace_method=mintrace_method,
    middle_level=middle_level,
)
print("✓ Reconciliation complete")


In [ ]:
# Coherence spot-check
result = spark.table(catalog + "." + schema + "." + use_case + "_reconciliation_output")
display(
    result.groupBy("hierarchy_level", "reconciliation_method")
    .agg(
        {"unique_id": "count", "y_base": "count"}
    )
)
print(f"Total rows: {result.count()}")
print(f"Levels: {[r[0] for r in result.select('hierarchy_level').distinct().collect()]}")
